[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-03-validators.ipynb#scrollTo=a0b1c2d3)

---
# Day 3 · Field Validators — @field_validator and Custom Logic
**certified-journeys / pydantic-certified** · Day 3 · Validation Logic

> **Goal for today:** By the end of this notebook you can write `@field_validator` methods in all four modes (`before`, `after`, `wrap`, `plain`), coerce incoming data, raise descriptive `ValueError`s, and apply one validator to multiple fields.

In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings

## Step 1 · @field_validator overview and the @classmethod requirement

In Pydantic v2, `@field_validator` replaces v1's `@validator`. The most important
syntactic rule: **every `@field_validator` method must also be a `@classmethod`**.
Omitting `@classmethod` does not raise an error but silently skips the validator
in some environments — always verify with a test.

```python
from pydantic import field_validator

class MyModel(BaseModel):
    field: str

    @field_validator('field')          # ① name the field(s)
    @classmethod                       # ② required — must come AFTER @field_validator
    def validate_field(cls, v):        # ③ cls, then the value
        ...                            # ④ return the (possibly transformed) value
        return v
```

The `mode` parameter controls *when* the validator fires relative to Pydantic's built-in
type coercion:

| Mode | Fires | Receives | Use for |
|------|-------|----------|---------|
| `'before'` | Before type coercion | Raw input | Coerce/normalise input |
| `'after'` (default) | After type coercion | Typed value | Logic on the final type |
| `'wrap'` | Around validation | Raw + call handler | Short-circuit or retry |
| `'plain'` | Instead of validation | Raw input | Fully custom type handling |

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

class SignupForm(BaseModel):
    email: str
    username: str

    # mode='after' (default): fires after str coercion, receives a str
    @field_validator('email', mode='after')
    @classmethod
    def normalise_email(cls, v: str) -> str:
        # Normalise email to lowercase — strip whitespace too
        return v.strip().lower()

    @field_validator('username', mode='after')
    @classmethod
    def normalise_username(cls, v: str) -> str:
        return v.strip()

# Email is normalised regardless of input case
form = SignupForm(email="  Alice@Example.COM  ", username="  alice  ")
print("email:",    form.email)     # alice@example.com
print("username:", form.username)  # alice

# Verify the validator actually fires — wrong value should still be a str
form2 = SignupForm(email="BOB@EXAMPLE.COM", username="Bob")
print("email2:", form2.email)  # bob@example.com

**What just happened?**

- The `@classmethod` decorator is **required** — it must appear *after* `@field_validator` in the decorator stack.
- `mode='after'` is the default; the validator receives an already-coerced value of the annotated type.
- **Return the value** — even if unchanged. Returning `None` replaces the field value with `None`.
- In Pydantic v2 the first argument is always `cls` (class method), not `self` — a key v1 → v2 migration change.

## Step 2 · mode='before' — coerce input before type validation

`mode='before'` fires **before** Pydantic attempts type coercion. The validator receives
the raw input value (which may not match the annotated type yet). This is ideal for:

- Splitting a comma-separated string into a `list`
- Stripping unwanted characters from a raw string
- Mapping legacy enum values to current ones

```
Input: "python,rust,go"  →  [before validator]  →  ["python", "rust", "go"]
                                                    ↓ Pydantic coerces to list[str]
```

In [ ]:
from pydantic import BaseModel, field_validator

class CourseEnrollment(BaseModel):
    student_name: str
    # Accepts either a real list OR a comma-separated string
    skills: list[str]

    @field_validator('skills', mode='before')
    @classmethod
    def coerce_skills_from_string(cls, v):
        """Accept comma-separated string and split it into a list."""
        if isinstance(v, str):
            # Split on comma, strip whitespace around each item, drop empty strings
            return [s.strip() for s in v.split(',') if s.strip()]
        # If already a list (or any other type), pass through to Pydantic's coercion
        return v

# Comma-separated string input
e1 = CourseEnrollment(student_name="Diana", skills="Python, Rust , Go")
print("From string:", e1.skills)      # ['Python', 'Rust', 'Go']

# Normal list input still works
e2 = CourseEnrollment(student_name="Evan", skills=["Python", "SQL"])
print("From list:  ", e2.skills)      # ['Python', 'SQL']

# Empty string produces empty list (empty strings dropped)
e3 = CourseEnrollment(student_name="Faye", skills="")
print("From empty: ", e3.skills)      # []

print("\nskills type:", type(e1.skills))  # <class 'list'>

**What just happened?**

- `mode='before'` gives you the **raw input** before Pydantic touches it — the value may be any Python type.
- After the `before` validator returns a `list`, Pydantic then validates each element against `list[str]`.
- **Always handle the non-string case** (pass through or raise) — otherwise a real `list` input would also try to split.
- This pattern is common when consuming CSV files, query strings, or legacy APIs.

## Step 3 · Raising ValueError with descriptive messages

Inside a `@field_validator`, raise `ValueError` (or `AssertionError`) to signal invalid data.
Pydantic wraps these in `ValidationError` — the message you provide becomes `err['msg']`.

**Good error messages tell the user what is wrong and what is expected:**
```python
# Bad  — no actionable info
raise ValueError("Invalid")

# Good — explains constraint and expected format
raise ValueError("must be a valid YYYY-MM-DD date string, got: {v!r}")
```

In [ ]:
import re
from pydantic import BaseModel, field_validator, ValidationError

class UserAccount(BaseModel):
    username: str
    password: str

    @field_validator('username', mode='after')
    @classmethod
    def validate_username(cls, v: str) -> str:
        # Must be 3–20 chars, alphanumeric + underscore
        if not re.fullmatch(r'[A-Za-z0-9_]{3,20}', v):
            raise ValueError(
                "username must be 3–20 characters and contain only letters, "
                f"digits, or underscores — got: {v!r}"
            )
        return v

    @field_validator('password', mode='after')
    @classmethod
    def validate_password_strength(cls, v: str) -> str:
        errors = []
        if len(v) < 8:
            errors.append("at least 8 characters")
        if not any(c.isupper() for c in v):
            errors.append("at least one uppercase letter")
        if not any(c.isdigit() for c in v):
            errors.append("at least one digit")
        if errors:
            raise ValueError("Password must contain: " + ", ".join(errors))
        return v

# Valid account
acct = UserAccount(username="grace_h", password="Secure99")
print("Valid account:", acct)

# Both fields fail — both errors reported at once
try:
    UserAccount(username="x!", password="weak")
except ValidationError as e:
    print(f"\n{e.error_count()} validation errors:")
    for err in e.errors():
        print(f"  [{err['loc'][0]}] {err['msg']}")

**What just happened?**

- `raise ValueError(...)` inside a validator becomes a `ValidationError` entry — the string becomes `err['msg']`.
- **Validators on different fields are independent** — both username and password errors are collected simultaneously.
- Include the offending value (`{v!r}`) in the error message to make debugging faster.
- You can also raise `AssertionError` (`assert condition, "message"`) — Pydantic treats it the same as `ValueError`.

## Step 4 · Applying one validator to multiple fields

`@field_validator` accepts multiple field names as positional arguments. The same
function is called once per listed field, with `v` being that field's value.

```python
@field_validator('first_name', 'last_name', 'middle_name', mode='after')
@classmethod
def clean_name(cls, v: str) -> str:
    return v.strip().title()
```

Use `info.field_name` to know which field is currently being validated when
you need field-specific branching.

In [ ]:
from pydantic import BaseModel, field_validator, ValidationInfo

class PersonName(BaseModel):
    first_name: str
    last_name: str
    middle_name: str = ""

    # One validator handles all three name fields
    @field_validator('first_name', 'last_name', 'middle_name', mode='after')
    @classmethod
    def normalise_name_part(
        cls,
        v: str,
        info: ValidationInfo,   # second arg gives field metadata
    ) -> str:
        field = info.field_name   # e.g. 'first_name', 'last_name', 'middle_name'

        # Allow empty string only for middle_name
        cleaned = v.strip().title()
        if not cleaned and field != 'middle_name':
            raise ValueError(f"{field} cannot be empty")
        return cleaned

# All three names are title-cased and stripped
p = PersonName(first_name="  alan  ", last_name="turing", middle_name="mathison")
print("first:",  p.first_name)   # Alan
print("last:",   p.last_name)    # Turing
print("middle:", p.middle_name)  # Mathison

# middle_name empty is fine; first_name empty is not
p2 = PersonName(first_name="Ada", last_name="Lovelace")  # middle defaults to ""
print("\nNo middle name:", p2)

from pydantic import ValidationError
try:
    PersonName(first_name="  ", last_name="Hopper")
except ValidationError as e:
    print("\nError:", e.errors()[0]['msg'])

**What just happened?**

- Listing multiple field names calls the **same function once per field** — no need to duplicate logic.
- `ValidationInfo` (second parameter, typed as `info: ValidationInfo`) exposes `info.field_name` so you can branch per field.
- The validator still fires independently for each field — if two fields fail, both errors appear in `ValidationError`.
- **Import `ValidationInfo` from `pydantic`**, not from `pydantic.fields` (which is where v1 kept it).

## Step 5 · mode='wrap' — wrap the full validation pipeline

`mode='wrap'` gives you a `handler` callable that represents the rest of Pydantic's
validation for that field. You call `handler(v)` to proceed normally, or skip it to
short-circuit. This is useful for:

- Catching validation errors from the pipeline and transforming them
- Providing a fallback value when validation fails
- Logging/auditing every field validation attempt

```python
@field_validator('field', mode='wrap')
@classmethod
def my_wrap(cls, v, handler):
    try:
        return handler(v)   # run normal validation
    except ValidationError:
        return fallback     # return fallback on failure
```

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

class Config(BaseModel):
    # If port is not a valid int or is out of range, fall back to 8080
    port: int

    @field_validator('port', mode='wrap')
    @classmethod
    def port_with_fallback(cls, v, handler):
        try:
            result = handler(v)       # run Pydantic's normal int coercion/validation
            if not (1 <= result <= 65535):
                raise ValueError(f"Port {result} out of valid range 1–65535")
            return result
        except (ValidationError, ValueError):
            # Fall back to default port instead of raising
            return 8080

# Valid port
c1 = Config(port=3000)
print("port:", c1.port)       # 3000

# String that can coerce to int
c2 = Config(port="5432")
print("port from string:", c2.port)  # 5432

# Invalid string → fallback to 8080
c3 = Config(port="invalid")
print("fallback port:", c3.port)     # 8080

# Out of range → fallback to 8080
c4 = Config(port=99999)
print("out of range:", c4.port)      # 8080

**What just happened?**

- `mode='wrap'` is the most powerful mode — you control *whether* Pydantic's validation runs and what to do with the result.
- The `handler` is a callable that runs the remaining validation pipeline and returns the validated value (or raises `ValidationError`).
- **Use sparingly** — returning a fallback silently hides bad input. Log or alert when doing so in production.
- `mode='plain'` is similar but skips Pydantic's type handling entirely; you are responsible for all conversion.

## Step 6 · Putting it together — a real-world User model

Combine `@field_validator` modes in a single model to handle normalisation (`before`),
business logic (`after`), and cross-field awareness (`ValidationInfo`).
This is the pattern you will use in FastAPI request bodies, CLI tools, and data pipelines.

In [ ]:
import re
from pydantic import BaseModel, Field, field_validator, ValidationError

class RegistrationRequest(BaseModel):
    email: str = Field(..., description="User email address")
    username: str = Field(..., min_length=3, max_length=30)
    # Accept comma-separated string OR list for roles
    roles: list[str] = Field(default_factory=list)
    referral_code: str = Field(default="", max_length=20)

    # before: coerce comma-separated string to list
    @field_validator('roles', mode='before')
    @classmethod
    def coerce_roles(cls, v):
        if isinstance(v, str):
            return [r.strip() for r in v.split(',') if r.strip()]
        return v

    # after: normalise email to lowercase
    @field_validator('email', mode='after')
    @classmethod
    def normalise_email(cls, v: str) -> str:
        v = v.strip().lower()
        # Basic format check — Day 5 introduces EmailStr for full RFC validation
        if '@' not in v or '.' not in v.split('@')[-1]:
            raise ValueError(f"Email does not look valid: {v!r}")
        return v

    # after: normalise username
    @field_validator('username', mode='after')
    @classmethod
    def normalise_username(cls, v: str) -> str:
        cleaned = v.strip().lower()
        if not re.fullmatch(r'[a-z0-9_]+', cleaned):
            raise ValueError(
                f"username may only contain lowercase letters, digits, and underscores — got {v!r}"
            )
        return cleaned

    # after: validate each role is a known value
    @field_validator('roles', mode='after')
    @classmethod
    def validate_roles(cls, v: list[str]) -> list[str]:
        VALID_ROLES = {"admin", "editor", "viewer"}
        invalid = [r for r in v if r not in VALID_ROLES]
        if invalid:
            raise ValueError(
                f"Unknown roles: {invalid}. Valid roles are: {sorted(VALID_ROLES)}"
            )
        return v

# Happy path with comma-separated roles
req = RegistrationRequest(
    email="  ALICE@Example.COM  ",
    username="Alice_2024",
    roles="admin, viewer",
    referral_code="FRIEND10"
)
print(req.model_dump())

# Multiple field failures
try:
    RegistrationRequest(
        email="not-an-email",
        username="BAD USER!",
        roles="superuser, admin",  # 'superuser' is unknown
    )
except ValidationError as e:
    print(f"\n{e.error_count()} errors:")
    for err in e.errors():
        print(f"  [{err['loc'][0]}] {err['msg']}")

**What just happened?**

- **Two validators on the same field** (`roles`) are ordered by mode: `before` fires first, then `after`.
- Validators on different fields run **independently** — all errors are collected before raising.
- The `before` validator on `roles` runs before Pydantic validates `list[str]`, so a comma string is safely split first.
- This pattern — normalise first (`before`), then assert business rules (`after`) — is the recommended v2 style.

In [ ]:
# Challenge: Build a PaymentRequest model with field validators
#
# 1. Define a PaymentRequest model with:
#    - amount: float        (must be > 0)
#    - currency: str        (must be one of: "USD", "EUR", "GBP", "JPY")
#    - card_number: str     (accept string with spaces/dashes, strip them in before validator)
#    - expiry: str          (format "MM/YY" — validate with regex in after validator)
#    - description: str = ""  (optional, strip whitespace)
#
# 2. Write these validators:
#    a. @field_validator('currency', mode='after') — uppercase + check allowed values
#    b. @field_validator('card_number', mode='before') — remove spaces and dashes
#    c. @field_validator('card_number', mode='after') — check it's 13–19 digits
#    d. @field_validator('expiry', mode='after') — validate MM/YY format with re.fullmatch
#    e. @field_validator('description', 'currency', mode='after') to strip whitespace
#       (use ValidationInfo to apply strip only to description)
#
# 3. Test with:
#    - valid: amount=19.99, currency="usd", card_number="4111 1111 1111 1111", expiry="12/26"
#    - invalid: currency="XYZ", card_number="1234", expiry="13/99"
#
# Your solution here


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `@field_validator` | Must always be paired with `@classmethod` |
| Signature | `def fn(cls, v)` or `def fn(cls, v, info: ValidationInfo)` |
| `mode='before'` | Fires before type coercion — receives raw input |
| `mode='after'` (default) | Fires after coercion — receives typed value |
| `mode='wrap'` | Receives raw input + `handler` callable — full control |
| `mode='plain'` | Replaces Pydantic's type handling entirely |
| Multiple fields | `@field_validator('a', 'b', 'c')` — same fn, called once per field |
| `ValidationInfo.field_name` | Identify current field inside a multi-field validator |
| `raise ValueError(...)` | Wraps into `ValidationError`; use descriptive messages |
| Execution order | `before` → type coercion → `after`; validators run per-field |

> **Tip:** In Pydantic v2, all `@field_validator` methods must be `@classmethod`. The first argument is `cls`, not `self`. Forgetting this silently ignores the validator in some IDE setups — always run a test to confirm it fires.

---
## What's next
**Day 4** → Model Validators and Cross-Field Logic — use `@model_validator` to validate relationships between fields (e.g. `end_date > start_date`) and access the entire model at once.

Mark Day 3 complete in your [tracker](../index.html).